# Batch ROI Fitting & Stitching
Per-tile fitting with pooled machine IRF, assembled into full-ROI maps.

**Workflow:**
1. Discover XLIF metadata files
2. For each ROI: fit all tiles independently (pooled machine IRF)
3. Assemble pixel maps with intensity-weighted overlap averaging
4. Derive global lifetime summary
5. Export intensity image, tau-coloured image (0–5 ns), and fit CSV

**Settings** — edit the config cell below, then `Run All`.

| Parameter | Value |
|-----------|-------|
| Exponentials | 3 |
| τ range | 0.145 – 45 ns |
| IRF | Machine IRF (pooled peak-aligned) |
| Tile overlap | Intensity-weighted averaging |
| Output | Intensity TIFF, τ-coloured PNG, global fit CSV |

In [ ]:
from pathlib import Path

# ── Edit these ────────────────────────────────────────────────────────────
XLIF_FOLDER      = Path("/Volumes/Lexar/20260316 Falcon data/xlifs")                      # folder containing .xlif files
PTU_FOLDER       = Path("/Volumes/Lexar/20260316 Falcon data/PTU.sptw")             # folder containing all .ptu files
OUTPUT_BASE      = Path("/Volumes/Lexar/20260316 Falcon data/Results")              # base output directory

# Fitting parameters
N_EXP            = 3
TAU_MIN_NS       = 0.145                 # ns  (matches 1.5 bins @ 97 ps)
TAU_MAX_NS       = 45.0                  # ns

# Lifetime colour image (intensity-weighted)
TAU_COLOUR_MIN   = 0.0                   # ns
TAU_COLOUR_MAX   = 5.0                   # ns  (typical biological range)

# Intensity display range
INTENSITY_DISPLAY_MIN = 0.0               # photons (for TIFF output)
INTENSITY_DISPLAY_MAX = None              # photons (None = auto-scale to max)

# Machine IRF — leave None to use FLIMKit default
MACHINE_IRF_PATH = None                  # e.g. Path("machine_irf.npy")

# DE optimiser settings (reduce for speed, increase for accuracy)
DE_POPSIZE       = 15
DE_MAXITER       = 1000
WORKERS          = -1                    # -1 = use all cores

# Skip files with fewer total photons than this
MIN_PHOTONS      = 3

# Rotation for Leica tiles
ROTATE_TILES     = True                  # 90° clockwise rotation

print(f"XLIF folder: {XLIF_FOLDER}")
print(f"PTU folder: {PTU_FOLDER}")
print(f"Output base: {OUTPUT_BASE}")

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from flimkit.PTU.reader      import PTUFile
from flimkit.FLIM.fitters    import fit_summed
from flimkit.configs         import (
    MACHINE_IRF_DEFAULT_PATH,
    MACHINE_IRF_FIT_BG, MACHINE_IRF_FIT_SIGMA, MACHINE_IRF_FIT_TAIL,
)

# Load machine IRF
_irf_path = Path(MACHINE_IRF_PATH) if MACHINE_IRF_PATH else Path(MACHINE_IRF_DEFAULT_PATH)
if not _irf_path.exists():
    raise FileNotFoundError(f"Machine IRF not found: {_irf_path}")

_machine_irf_raw = np.load(str(_irf_path)).ravel().astype(float)
_machine_irf_raw = np.maximum(_machine_irf_raw, 0.0)
_machine_irf_raw /= _machine_irf_raw.sum()
PI_MACHINE = int(np.argmax(_machine_irf_raw))

print(f"Machine IRF: {_irf_path.name}  |  {len(_machine_irf_raw)} bins  |  peak bin {PI_MACHINE}")
print(f"FIT_BG={MACHINE_IRF_FIT_BG}  FIT_SIGMA={MACHINE_IRF_FIT_SIGMA}  HAS_TAIL={MACHINE_IRF_FIT_TAIL}")
print(f"n_exp={N_EXP}  τ=[{TAU_MIN_NS}, {TAU_MAX_NS}] ns")
print(f"Colour range: {TAU_COLOUR_MIN}–{TAU_COLOUR_MAX} ns")

def get_irf(peak_bin: int, n_bins: int) -> np.ndarray:
    """Get machine IRF, padded/cropped to n_bins and peak-aligned to peak_bin."""
    irf = _machine_irf_raw.copy()
    if irf.size > n_bins:
        irf = irf[:n_bins]
    elif irf.size < n_bins:
        irf = np.pad(irf, (0, n_bins - irf.size))
    shift = peak_bin - PI_MACHINE
    if shift:
        irf = np.roll(irf, shift)
    s = irf.sum()
    return irf / s if s > 0 else irf

print("\nImports & IRF OK ✓")

In [ ]:
# Discover XLIF files and infer PTU basenames
xlif_files = sorted(XLIF_FOLDER.glob("*.xlif"))
print(f"Found {len(xlif_files)} XLIF file(s):")
for x in xlif_files:
    print(f"  {x.name}")

if not xlif_files:
    print("\nNo XLIF files found. Checking subdirectories...")
    xlif_files = sorted(XLIF_FOLDER.rglob("*.xlif"))
    if xlif_files:
        print(f"Found {len(xlif_files)} XLIF file(s) in subdirectories:")
        for x in xlif_files:
            print(f"  {x.relative_to(XLIF_FOLDER)}")

In [4]:
from flimkit.PTU.stitch import fit_flim_tiles
from flimkit.FLIM.assemble import assemble_tile_maps, derive_global_tau, save_assembled_maps
from flimkit.utils.lifetime_image import make_lifetime_image
import argparse
import tifffile
import gc
from tqdm.notebook import tqdm

# Prepare CSV for streaming results
csv_path = OUTPUT_BASE / "batch_roi_fit_global_summary.csv"
csv_header_written = False

for xlif_idx, xlif_path in enumerate(xlif_files):
    xlif_path = Path(xlif_path)
    ptu_basename = xlif_path.stem  # e.g., "R 2"
    roi_clean = ptu_basename.replace(' ', '_')
    
    output_dir = OUTPUT_BASE / roi_clean
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n{'='*60}")
    print(f"  ROI {xlif_idx + 1}/{len(xlif_files)}: {ptu_basename}")
    print(f"{'='*60}")
    
    try:
        # Build full args like interactive.py does
        fit_args = argparse.Namespace(
            nexp              = N_EXP,
            tau_min           = TAU_MIN_NS,
            tau_max           = TAU_MAX_NS,
            optimizer         = 'de',
            restarts          = 1,
            de_population     = DE_POPSIZE,
            de_maxiter        = DE_MAXITER,
            workers           = WORKERS,
            binning           = 1,
            min_photons       = MIN_PHOTONS,
            machine_irf       = str(MACHINE_IRF_PATH) if MACHINE_IRF_PATH else str(_irf_path),
            cost_function     = 'poisson',
            tau_display_min   = TAU_COLOUR_MIN,
            tau_display_max   = TAU_COLOUR_MAX,
            irf_xlsx_dir      = None,
            irf_xlsx_map      = None,
            ptu_basename      = ptu_basename,
            xlif              = str(xlif_path),
            ptu_dir           = str(PTU_FOLDER),
            output_dir        = str(output_dir),
        )
        
        print(f"  [1/5] Fitting tiles...")
        tile_results, canvas_height, canvas_width = fit_flim_tiles(
            xlif_path     = xlif_path,
            ptu_dir       = PTU_FOLDER,
            output_dir    = output_dir,
            args          = fit_args,
            ptu_basename  = ptu_basename,
            rotate_tiles  = ROTATE_TILES,
            verbose       = False,
        )
        
        if not tile_results:
            print(f"  ⚠ No tiles fitted")
            result = {'roi': ptu_basename, 'status': 'No tiles fitted'}
            df_row = pd.DataFrame([result])
            if not csv_header_written:
                df_row.to_csv(csv_path, index=False)
                csv_header_written = True
            else:
                df_row.to_csv(csv_path, mode='a', index=False, header=False)
            del tile_results, df_row
            gc.collect()
            continue
        
        print(f"  [2/5] Assembling canvas ({len(tile_results)} tiles)...")
        canvas = assemble_tile_maps(
            tile_results   = tile_results,
            canvas_height  = canvas_height,
            canvas_width   = canvas_width,
            n_exp          = N_EXP,
        )
        del tile_results
        gc.collect()
        
        print(f"  [3/5] Computing global summary...")
        global_summary = derive_global_tau(canvas, n_exp=N_EXP)
        
        tau_mean = global_summary.get('tau_mean_amp_global_ns', float('nan'))
        n_pixels = global_summary.get('n_pixels_fitted', 0)
        print(f"        τ_mean = {tau_mean:.4f} ns  |  {n_pixels:,} pixels")
        
        print(f"  [4/5] Saving maps & images...")
        save_assembled_maps(
            canvas        = canvas,
            global_summary= global_summary,
            output_dir    = output_dir,
            roi_name      = roi_clean,
            n_exp         = N_EXP,
            tau_display_min = 0.0,
            tau_display_max = 10.0,
            intensity_display_min = INTENSITY_DISPLAY_MIN,
            intensity_display_max = INTENSITY_DISPLAY_MAX,
        )
        
        make_lifetime_image(
            canvas       = canvas,
            output_dir   = output_dir,
            roi_name     = roi_clean,
            tau_min_ns   = TAU_COLOUR_MIN,
            tau_max_ns   = TAU_COLOUR_MAX,
            smooth_sigma_px = 2.0,
            intensity_percentile_lo = 0.5,
            intensity_percentile_hi = 99.5,
            gamma        = 0.5,
            verbose      = False,
        )
        
        print(f"  [5/5] Writing CSV...")
        result = {'roi': ptu_basename, 'status': 'OK', **global_summary}
        df_row = pd.DataFrame([result])
        if not csv_header_written:
            df_row.to_csv(csv_path, index=False)
            csv_header_written = True
        else:
            df_row.to_csv(csv_path, mode='a', index=False, header=False)
        
        print(f"  ✓ {ptu_basename} complete")
        
    except Exception as e:
        print(f"  ✗ {ptu_basename} failed: {type(e).__name__}: {str(e)[:100]}")
        result = {'roi': ptu_basename, 'status': f'ERROR: {type(e).__name__}'}
        df_row = pd.DataFrame([result])
        if not csv_header_written:
            df_row.to_csv(csv_path, index=False)
            csv_header_written = True
        else:
            df_row.to_csv(csv_path, mode='a', index=False, header=False)
    
    finally:
        # Cleanup
        for var in ['canvas', 'global_summary', 'result', 'df_row', 'tile_results', 'fit_args']:
            try:
                del locals()[var]
            except KeyError:
                pass
        gc.collect()

print(f"\n{'='*60}")
print(f"✓ All ROIs processed")
print(f"✓ CSV: {csv_path}")
print(f"{'='*60}")

In [ ]:
# Load and display the final summary CSV
csv_path = OUTPUT_BASE / "batch_roi_fit_global_summary.csv"
df_results = pd.read_csv(csv_path)

print(f"\nGlobal summary CSV → {csv_path}")
print(f"Shape: {df_results.shape[0]} ROIs × {df_results.shape[1]} columns\n")
print(df_results.to_string())

In [ ]:
import matplotlib.pyplot as plt

# Summary statistics
ok = df_results[df_results.status == "OK"].copy() if 'status' in df_results.columns else df_results.copy()

if len(ok) > 0:
    print("\n── Global Lifetime Summary Statistics ───────────────────────")
    stat_cols = [
        'tau_mean_amp_global_ns', 'tau_std_amp_global_ns',
        'tau_median_amp_global_ns',
        'n_pixels_fitted'
    ]
    stat_cols = [c for c in stat_cols if c in ok.columns]
    if stat_cols:
        print(ok[stat_cols].describe().round(4).to_string())
    
    # Plot tau distributions if multiple ROIs
    if len(ok) > 1 and 'tau_mean_amp_global_ns' in ok.columns:
        fig, axes = plt.subplots(1, min(3, len(ok)+1), figsize=(4*min(3, len(ok)+1), 3.5))
        if not isinstance(axes, np.ndarray):
            axes = [axes]
        
        if 'tau_mean_amp_global_ns' in ok.columns:
            axes[0].bar(ok['roi'], ok['tau_mean_amp_global_ns'],
                        color='#028090', edgecolor='white', alpha=0.85, width=0.6)
            axes[0].set_ylabel('τ mean (ns)')
            axes[0].set_title('τ mean (amplitude-weighted)')
            axes[0].grid(True, alpha=0.3, axis='y')
            axes[0].tick_params(axis='x', rotation=45)
        
        if len(axes) > 1 and 'tau_std_amp_global_ns' in ok.columns:
            axes[1].bar(ok['roi'], ok['tau_std_amp_global_ns'],
                        color='#e74c3c', edgecolor='white', alpha=0.85, width=0.6)
            axes[1].set_ylabel('τ std (ns)')
            axes[1].set_title('τ std (amplitude-weighted)')
            axes[1].grid(True, alpha=0.3, axis='y')
            axes[1].tick_params(axis='x', rotation=45)
        
        if len(axes) > 2 and 'n_pixels_fitted' in ok.columns:
            axes[2].bar(ok['roi'], ok['n_pixels_fitted'],
                        color='#2ecc71', edgecolor='white', alpha=0.85, width=0.6)
            axes[2].set_ylabel('# fitted pixels')
            axes[2].set_title('Fitted pixel count')
            axes[2].grid(True, alpha=0.3, axis='y')
            axes[2].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.savefig(OUTPUT_BASE / "batch_roi_fit_summary_plots.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"\nPlot saved → {OUTPUT_BASE / 'batch_roi_fit_summary_plots.png'}")
else:
    print("No successful ROI fits to summarize.")